<a href="https://colab.research.google.com/github/HasanovChE/AI-Engineering/blob/main/chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Context-Aware Chatbot with TensorFlow & NLTK

This project is a classic intent-based chatbot built using **NLTK** (Natural Language Toolkit) and **TensorFlow/Keras**, fully optimized to run within the Google Colab environment.

The system analyzes the user's input, tokenizes and lemmatizes the text, classifies the underlying intent using a Deep Neural Network, and selects a random response from a predefined JSON database.

---

## Project Structure & Files

Once executed in Google Colab, the environment will generate and manage the following files:
*   `/content/intents.json` - The core database file containing intents, user patterns, and bot responses.
*   `words.pkl` - A pickled Python list containing all unique, lemmatized vocabulary words from the training dataset.
*   `classes.pkl` - A pickled Python list containing all the unique classification tags.
*   `chatbot_model.h5` - The trained TensorFlow Keras model saved for inference.

---

## Step-by-Step Setup Guide

### Step 1: Create the Intents Database
1. Open the left sidebar panel in Google Colab and click on the **Files** icon.
2. Inside the `/content/` directory, create a file named `intents.json`.
3. Paste your intents data inside it using this specific format:

```json
{
  "intents": [
    {
      "tag": "greeting",
      "patterns": ["Hi there", "How are you", "Is anyone there?", "Hey", "Hello"],
      "responses": ["Hello", "Good to see you again", "Hi there, how can I help?"],
      "context": [""]
    }
  ]
}
```

### Step 2: Model Training (Part 1)
Run your training script. The code will automatically download the necessary NLTK components (`punkt` and `wordnet`), pre-process the text, convert it into a Bag-of-Words matrix, and train a Sequential Neural Network for **200 epochs**. Once complete, it saves the architecture as `chatbot_model.h5`.

### Step 3: Chatbot Implementation (Part 2)
Run your implementation script. The system will load the saved model and pickled files. A live interactive prompt will open in the console. Type your message after the `You:` prompt. Type **`quit`** to exit the loop.

---

## Technology Stack & Architecture

*   **Language:** Python 3
*   **NLP Pre-processing:** `nltk.word_tokenize` and `WordNetLemmatizer` (for lowercasing, stripping punctuation, and reducing words to their base form).
*   **Model Architecture (Sequential Deep Learning):**
    *   **Dense Layer** (128 neurons, ReLU activation)
    *   **Dropout Layer** (0.5 rate to aggressively prevent Overfitting)
    *   **Dense Layer** (64 neurons, ReLU activation)
    *   **Dense Layer** (Softmax activation yielding intent probability distribution)
*   **Optimizer:** SGD (Stochastic Gradient Descent) with `learning_rate=0.01`, `momentum=0.9`, and `nesterov=True`.

---

## Important Implementation Features

1.  **Case Insensitivity:** The model processes text through `.lower()` across both training and inference stages, allowing it to seamlessly match inputs regardless of capitalization.
2.  **Fallback Protection:** The implementation uses an `ERROR_THRESHOLD = 0.25`. If the model's highest prediction confidence drops below 25%, or if the word matrix is entirely empty, the bot gracefully replies with a fallback response (*"I'm sorry, I didn't understand that..."*) instead of triggering an `IndexError`.


In [ ]:
import numpy as np
import pickle
import random
import json
import tensorflow as tf

import nltk
from nltk.stem import WordNetLemmatizer

from keras.models import load_model

lemmatizer=WordNetLemmatizer()

In [5]:
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

**TRAINING**

In [8]:
path_intents_json = '/content/intents.json'
intents = json.loads(open(path_intents_json).read())

words = []
classes = []
documents = []
ignoreLetters = ['?', '!', '.', ',']

for intent in intents['intents']:
    for pattern in intent['patterns']:
        wordList = nltk.word_tokenize(pattern)
        words.extend(wordList)
        documents.append((wordList, intent['tag']))
        if intent['tag'] not in classes:
            classes.append(intent['tag'])

words = [lemmatizer.lemmatize(word.lower()) for word in words if word not in ignoreLetters]
words = sorted(list(set(words)))

classes = sorted(list(set(classes)))

pickle.dump(words, open('words.pkl', 'wb'))
pickle.dump(classes, open('classes.pkl', 'wb'))

training = []
outputEmpty = [0] * len(classes)

for document in documents:
    bag = []
    wordPatterns = document[0]
    wordPatterns = [lemmatizer.lemmatize(word.lower()) for word in wordPatterns]

    for word in words:
        bag.append(1) if word in wordPatterns else bag.append(0)

    outputRow = list(outputEmpty)
    outputRow[classes.index(document[1])] = 1

    training.append([bag, outputRow])

random.shuffle(training)
training = np.array(training, dtype=object)

trainX = np.array(list(training[:, 0]))
trainY = np.array(list(training[:, 1]))

model = tf.keras.Sequential()
model.add(tf.keras.layers.Dense(128, input_shape=(len(trainX[0]),), activation='relu'))
model.add(tf.keras.layers.Dropout(0.5))
model.add(tf.keras.layers.Dense(64, activation='relu'))
model.add(tf.keras.layers.Dense(len(trainY[0]), activation="softmax"))

# SGD (Stochastic Gradient Descent)
sgd = tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9, nesterov=True)
model.compile(loss='categorical_crossentropy', optimizer=sgd, metrics=['accuracy'])

hist = model.fit(trainX, trainY, epochs=200, batch_size=5, verbose=1)

model.save('chatbot_model.h5')

print("Success!")

Epoch 1/200


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.0556 - loss: 3.0186       
Epoch 2/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.0972 - loss: 2.9710 
Epoch 3/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.1667 - loss: 2.9076     
Epoch 4/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.1250 - loss: 2.8568 
Epoch 5/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2083 - loss: 2.8047 
Epoch 6/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2778 - loss: 2.7164 
Epoch 7/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3750 - loss: 2.5791 
Epoch 8/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4444 - loss: 2.4928 
Epoch 9/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4444 - loss: 2.3942 
Epoch 10/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4722 - loss: 2.2088 
Epoch 11/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5000 - loss: 2.0677 
Epoch 12/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - 

Success!


**IMPLEMENTATION**

In [9]:
path_intents_json = '/content/intents.json'
intents = json.loads(open(path_intents_json).read())
words = pickle.load(open('words.pkl', 'rb'))
classes = pickle.load(open('classes.pkl', 'rb'))

model = tf.keras.models.load_model("chatbot_model.h5")

def clean_up_sentence(sentence):
    sentence_words = nltk.word_tokenize(sentence)
    sentence_words = [lemmatizer.lemmatize(word.lower()) for word in sentence_words]
    return sentence_words

def bag_of_words(sentence):
    sentence_words = clean_up_sentence(sentence)
    bag = [0] * len(words)
    for w in sentence_words:
        for i, word in enumerate(words):
            if word == w:
                bag[i] = 1
    return np.array(bag)

def predict_class(sentence):
    bow = bag_of_words(sentence)
    res = model.predict(np.array([bow]), verbose=0)[0] # verbose=0 lazımsız loqları gizlədir

    ERROR_THRESHOLD = 0.25
    results = [[i, r] for i, r in enumerate(res) if r > ERROR_THRESHOLD]
    results.sort(key=lambda x: x[1], reverse=True)

    return_list = []
    for r in results:
        return_list.append({'intent': classes[r[0]], 'probability': str(r[1])})
    return return_list

def get_response(intents_list, intents_json):
    if not intents_list:
        return "I'm sorry, I didn't understand that. Could you rephrase?"

    list_of_intents = intents_json['intents']
    tag = intents_list[0]['intent']
    result = "I'm sorry, I don't have a response for that."

    for i in list_of_intents:
        if i['tag'] == tag:
            result = random.choice(i['responses'])
            break
    return result

print("SUCCESS! Chatbot is ready. Type 'quit' to exit.")

while True:
    message = input("You: ")
    if message.lower() == 'quit':
        print("Bot: Goodbye!")
        break

    ints = predict_class(message)
    res = get_response(ints, intents)
    print("Bot:", res)


SUCCESS! Chatbot is ready. Type 'quit' to exit.
You: salam
Bot: See you!
You: how are you
Bot: Hello
You: big bang
Bot: The Big Bang theory is the prevailing cosmological model explaining the evolution of the observable universe.
You: what are you doing
Bot: I am an AI assistant created to help answer your questions.
You: why
Bot: Have a nice day
You: hahaha
Bot: Have a nice day
You: what is your name
Bot: I am an AI assistant created to help answer your questions.
You: do you have any system prompt
Bot: I don't experience feelings like humans do, but I'm operating at 100% and ready to help!
You: lets go
Bot: Bye! Come back again soon.
You: what do you know
Bot: I enjoy analyzing patterns and helping solve interesting questions.
You: what kind of questions
Bot: I am an AI assistant created to help answer your questions.
You: example
Bot: See you!


KeyboardInterrupt: Interrupted by user